# Iniciando o Spark



In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
from pyspark.sql import SparkSession
import os
import pytz
from datetime import datetime

spark = SparkSession.builder.appName("tabelas").getOrCreate()

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [ ]:
spark

# Instalando bibliotecas

In [ ]:

import os
import sys
import pytz
import numpy as np
import datetime
from pyspark.sql import SparkSession
from pyspark.sql import SQLContext
from pyspark.sql.functions import udf,split, lpad, concat_ws,to_timestamp,col,coalesce
from datetime import datetime
from datetime import timedelta
from datetime import date
from dateutil.relativedelta import relativedelta
from pyspark.sql.types import *
from pyspark.sql.functions import count, avg, to_date

#Configuração do pipeline( contem configuração do spark e os paths para serem alterados)


In [ ]:
# caminhos dos arquivos

PATH_TABELA_BUREAU ="/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_score_bureau_movel_full"
PATH_TABELA_CADASTRO = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_dados_cadastrais"
PATH_BASE_TELCO       = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/base_telco"
PATH_BASE_RECARGA     = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/bases_recarga/BI_FP_ASS_RECARGA_CMV_NOVA"
PATH_BASE_BOOK_ATRASO = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/book_atraso/dados_faturamento"
PATH_BASE_BOOK_PAGAMENTO ="/content/gdrive/MyDrive/Raw Hackathon PoD 2025/book_pagamento/dados_pagamento"

BASE_PATH_RECARGA = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/bases_recarga"

PATH_DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO": f"{BASE_PATH_RECARGA}/BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": f"{BASE_PATH_RECARGA}/BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": f"{BASE_PATH_RECARGA}/BI_DIM_INSTITUICAO.csv",
    "PLATAFORMA": f"{BASE_PATH_RECARGA}/BI_DIM_PLATAFORMA.csv",
    "PROMOCAO": f"{BASE_PATH_RECARGA}/BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": f"{BASE_PATH_RECARGA}/BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": f"{BASE_PATH_RECARGA}/BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": f"{BASE_PATH_RECARGA}/BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": f"{BASE_PATH_RECARGA}/BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": f"{BASE_PATH_RECARGA}/BI_DIM_TIPO_RECARGA.csv",
}


PATH_DIMENSOES_BOOK_ATRASO = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/book_atraso/BI_DIM_TIPO_FATURAMENTO.csv"

In [ ]:
# Csvs da tabela recarga

BASE_PATH_RECARGA = "/content/gdrive/MyDrive/Raw Hackathon PoD 2025/bases_recarga"

DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO_CREDITO": "BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": "BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": "BI_DIM_INSTITUICAO.csv",
    "PLANO_PRECO": "BI_DIM_PLANO_PRECO.csv",
    "PLATAFORMA": "BI_DIM_PLATAFORMA.csv",
    "PROMOCAO_CREDITO": "BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": "BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": "BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": "BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": "BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": "BI_DIM_TIPO_RECARGA.csv",
}


dfs_dim_recarga = {}

for nome_dim, arquivo in DIMENSOES_RECARGA.items():
    path = f"{BASE_PATH_RECARGA}/{arquivo}"

    dfs_dim_recarga[nome_dim] = (
        spark.read
        .option("header", True)
        .option("sep", ",")
        .option("inferSchema", True)
        .csv(path)
    )

df_CANAL_AQUISICAO_CREDITO = dfs_dim_recarga["CANAL_AQUISICAO_CREDITO"]
df_FORMA_PAGAMENTO        = dfs_dim_recarga["FORMA_PAGAMENTO"]
df_INSTITUICAO            = dfs_dim_recarga["INSTITUICAO"]
df_PLANO_PRECO            = dfs_dim_recarga["PLANO_PRECO"]
df_PLATAFORMA             = dfs_dim_recarga["PLATAFORMA"]
df_PROMOCAO_CREDITO       = dfs_dim_recarga["PROMOCAO_CREDITO"]
df_STATUS_PLATAFORMA      = dfs_dim_recarga["STATUS_PLATAFORMA"]
df_TECNOLOGIA              = dfs_dim_recarga["TECNOLOGIA"]
df_TIPO_CREDITO           = dfs_dim_recarga["TIPO_CREDITO"]
df_TIPO_INSERCAO          = dfs_dim_recarga["TIPO_INSERCAO"]
df_TIPO_RECARGA           = dfs_dim_recarga["TIPO_RECARGA"]

In [ ]:
# cadastro
df_cadastro = spark.read.parquet(PATH_TABELA_CADASTRO)
df_cadastro.createOrReplaceTempView("df_cadastro")

# bureau
df_bureau = spark.read.parquet(PATH_TABELA_BUREAU)
df_bureau.createOrReplaceTempView("df_bureau")

# telco
df_base_telco = spark.read.parquet(PATH_BASE_TELCO)
df_base_telco.createOrReplaceTempView("df_base_telco")

# recarga
df_recarga = spark.read.parquet(PATH_BASE_RECARGA)
df_recarga.createOrReplaceTempView("df_recarga")

# book atraso
df_book_atraso = spark.read.parquet(PATH_BASE_BOOK_ATRASO)
df_book_atraso.createOrReplaceTempView("df_book_atraso")

# book pagamento
df_book_pagamento = spark.read.parquet(PATH_BASE_BOOK_PAGAMENTO)
df_book_pagamento.createOrReplaceTempView("df_book_pagamento")


# Carregamento das bases de dados para unificação das tabelas

As bases de dados serão unificadas utilizando o **NUM_CPF** e a **SAFRA** como chaves de relacionamento, garantindo a correta associação dos registros ao longo do tempo e evitando duplicidades.

Cada tabela terá suas variáveis identificadas por um **prefixo específico**, permitindo que a área de dados identifique claramente a **origem de cada variável** após a unificação das bases.

Essa estratégia assegura maior **consistência temporal**, **rastreabilidade dos dados**, **organização do modelo** e **facilidade na manutenção e nas análises analíticas**.


In [ ]:
# base bureau
base_score_bureau_movel_full = spark.read.parquet(PATH_TABELA_BUREAU)
base_score_bureau_movel_full.createOrReplaceTempView("base_score_bureau")

In [ ]:
# Alterando o datatype correto da tabela
base_score_bureau_movel_full.createOrReplaceTempView("base_score_bureau_movel_full")

# Faz o cast e renomeia colunas
df_bureau = spark.sql("""
    SELECT
        -- SAFRA convertida para DATE (primeiro dia do mês)
        CAST(CONCAT(SUBSTRING(SAFRA, 1, 4), '-', SUBSTRING(SAFRA, 5, 2), '-01') AS DATE) AS SAFRA,

        -- Ano e Mês extraídos da SAFRA
        CAST(SUBSTRING(SAFRA, 1, 4) AS INT) AS Ano,
        CAST(SUBSTRING(SAFRA, 5, 2) AS INT) AS Mes,

        CAST(FLAG_INSTALACAO AS BOOLEAN) AS IsInstallation,
        CAST(PROD AS STRING) AS ProductDescription,
        CAST(flag_mig2 AS STRING) AS ProductMigration,
        CAST(SCORE_01 AS FLOAT) AS Score01,
        CAST(SCORE_02 AS FLOAT) AS Score02,
        CAST(FPD AS INT) AS FPD,
        CAST(NUM_CPF AS STRING) AS NUM_CPF
    FROM base_score_bureau_movel_full
""")

# colocar indentificador nas variaveis da tabela bureau
df_bureau_01 = df_bureau

for c in df_bureau.columns:
    df_bureau_01 = df_bureau_01.withColumnRenamed(c, f"B_BUREAU_{c}")


In [ ]:
#base dados cadastrias
base_dados_cadastrais = spark.read.parquet(PATH_TABELA_CADASTRO)
base_dados_cadastrais.createOrReplaceTempView("base_dados_cadastrais")

In [ ]:
# Criar TempView
base_dados_cadastrais.createOrReplaceTempView("base_dados_cadastrais")

# Gerar as colunas var_02 até var_25 como STRING
var_columns_sql = ",".join([f"CAST(var_{i:02d} AS STRING) AS Var{i}" for i in range(2, 26)])


df_cadastrais = spark.sql(f"""
    SELECT
        CAST(NUM_CPF AS STRING) AS NUM_CPF,

        -- SAFRA convertida para DATE (primeiro dia do mês)
        CAST(CONCAT(SUBSTRING(SAFRA,1,4), '-', SUBSTRING(SAFRA,5,2), '-01') AS DATE) AS SAFRA,

        -- Ano e Mês extraídos da SAFRA
        CAST(SUBSTRING(SAFRA,1,4) AS INT) AS SAFRA_ANO,
        CAST(SUBSTRING(SAFRA,5,2) AS INT) AS SAFRA_MES,

        CAST(FLAG_INSTALACAO AS BOOLEAN) AS IsSetup,
        CAST(FPD AS BOOLEAN) AS IsFPD,
        CAST(PROD AS STRING) AS ProductDescription,
        CAST(flag_mig2 AS STRING) AS ProductMigration,
        CAST(STATUSRF AS STRING) AS StatusRF,

        -- Corrigir formato dd/MM/yyyy para DATE
        to_date(DATADENASCIMENTO, 'dd/MM/yyyy') AS DataNascimento,

        {var_columns_sql},

        CAST(CEP_3_digitos AS STRING) AS CEP_3_digitos
    FROM base_dados_cadastrais
""")

# colocar indentificador nas variaveis da tabela cadastro
df_cadastrais_01 = df_cadastrais

for c in df_cadastrais.columns:
    df_cadastrais_01 = df_cadastrais_01.withColumnRenamed(c, f"B_CADASTRO_{c}")


In [ ]:
# Fazendo o join das duas tabelas
df_bureau_01.createOrReplaceTempView("bureau")
df_cadastrais_01.createOrReplaceTempView("cadastro")

df_principal = spark.sql("""
SELECT
    b.*,
    c.*,

    -- CPF unificado
    COALESCE(b.B_BUREAU_NUM_CPF, c.B_CADASTRO_NUM_CPF) AS NUM_CPF

FROM bureau b
LEFT JOIN cadastro c
    ON b.B_BUREAU_NUM_CPF = c.B_CADASTRO_NUM_CPF
    AND b.B_BUREAU_SAFRA = c.B_CADASTRO_SAFRA
""")


In [ ]:
# retirar a coluna de cpfs das duas bases para evitar confusão com os dados
df_principal = df_principal.drop(
    "B_BUREAU_NUM_CPF",
    "B_CADASTRO_NUM_CPF"
)


In [ ]:
# Pega todas as colunas
cols = df_principal.columns

# Reordena colocando NUM_CPF primeiro
new_order = ["NUM_CPF"] + [c for c in cols if c != "NUM_CPF"]

# Reaplica a ordem
df_principal = df_principal.select(*new_order)


# Tabela telco com base principal

In [ ]:
# carregando a tabela  base_telco
df_base_telco =  spark.read.parquet(PATH_BASE_TELCO)
df_base_telco.createOrReplaceTempView("df_base_telco")

In [ ]:
# alterando o datatype, criando coluna safra
df_base_telco = spark.sql(f"""
    SELECT
            CAST(NUM_CPF AS STRING) AS NUM_CPF,
            CAST(SAFRA AS INT) AS SAFRA,

            CAST(SUBSTRING(CAST(SAFRA AS STRING), 1, 4) AS INT) AS SAFRA_ANO,
            CAST(SUBSTRING(CAST(SAFRA AS STRING), 5, 2) AS INT) AS SAFRA_MES,

            TO_DATE(
                CONCAT(
                    SUBSTRING(CAST(SAFRA AS STRING), 1, 4), '-',
                    SUBSTRING(CAST(SAFRA AS STRING), 5, 2), '-01'
                )
            ) AS SAFRA_REFERENCIA,

            CAST(FLAG_INSTALACAO AS BOOLEAN) AS IsSetup,
            CAST(FPD AS BOOLEAN) AS IsFPD,
            CAST(PROD AS STRING) AS ProductDescription,
            CAST(flag_mig2 AS STRING) AS ProductMigration,

            {",".join([f"CAST(var_{i} AS FLOAT) AS Var{i}" for i in range(26,94)])}

        FROM df_base_telco
""")

# colocar indentificador nas variaveis da tabela telco
df_base_telco = df_base_telco

for c in df_base_telco.columns:
    df_base_telco = df_base_telco.withColumnRenamed(c, f"B_TELCO_{c}")

In [ ]:
# tabelas temporarias
df_base_telco.createOrReplaceTempView("telco")
df_principal.createOrReplaceTempView("df_principal")

# variáveis telco
vars_telco = ",\n    ".join(
    [f"t.B_TELCO_Var{i} AS B_TELCO_Var{i}" for i in range(26,94)]
)

df_principal_telco = spark.sql(f"""
SELECT
    -- base principal (bureau + cadastro)
    b.*,

    -- TELCO (snapshot na mesma safra)
    t.B_TELCO_SAFRA,
    t.B_TELCO_SAFRA_ANO,
    t.B_TELCO_SAFRA_MES,
    t.B_TELCO_IsSetup,
    t.B_TELCO_IsFPD,
    t.B_TELCO_ProductDescription,
    t.B_TELCO_ProductMigration,

    -- variáveis telco
    {vars_telco}

FROM df_principal b
LEFT JOIN telco t
    ON b.NUM_CPF = t.B_TELCO_NUM_CPF
   AND b.B_BUREAU_SAFRA = t.B_TELCO_SAFRA_REFERENCIA
""")

In [ ]:
# carregar tabela recarga
df_base_recarga = spark.read.parquet(PATH_BASE_RECARGA)
df_base_recarga.createOrReplaceTempView("df_base_recarga")

In [ ]:
# Mudando o datatype
df_base_recarga = spark.sql("""
    SELECT
        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        CAST(DW_NUM_NTC AS STRING) AS DW_NUM_NTC,
        CAST(DW_NUM_CLIENTE AS STRING) AS DW_NUM_CLIENTE,
        CAST(DAT_INSERCAO_CREDITO AS STRING) AS DAT_INSERCAO_CREDITO,
        CAST(HOR_INSERCAO_CREDITO AS STRING) AS HOR_INSERCAO_CREDITO,
        CAST(COD_TECNOLOGIA_DW AS STRING) AS COD_TECNOLOGIA_DW,
        CAST(COD_CANAL_AQUISICAO AS INT) AS COD_CANAL_AQUISICAO,
        CAST(COD_TIPO_CREDITO AS STRING) AS COD_TIPO_CREDITO,
        CAST(COD_PROMOCAO AS INT) AS COD_PROMOCAO,
        CAST(VAL_CREDITO_INSERIDO AS FLOAT) AS VAL_CREDITO_INSERIDO,
        CAST(VAL_BONUS AS FLOAT) AS VAL_BONUS,
        CAST(VAL_REAL AS FLOAT) AS VAL_REAL,
        CAST(COD_PLATAFORMA_ATU AS STRING) AS COD_PLATAFORMA_ATU,
        CAST(COD_STATUS_PLATAFORMA AS STRING) AS COD_STATUS_PLATAFORMA,
        CAST(IND_METODO_PAGAMENTO AS STRING) AS IND_METODO_PAGAMENTO,
        CAST(DW_PLANO_TARIFACAO AS INT) AS DW_PLANO_TARIFACAO,
        CAST(DW_TIPO_RECARGA AS INT) AS DW_TIPO_RECARGA,
        CAST(DW_TIPO_INSERCAO AS INT) AS DW_TIPO_INSERCAO,
        CAST(DW_FORMA_PAGAMENTO AS INT) AS DW_FORMA_PAGAMENTO,
        CAST(DW_INSTITUICAO AS INT) AS DW_INSTITUICAO,
        CAST(COD_GRUPO_CARTAO AS STRING) AS COD_GRUPO_CARTAO,
        CAST(DSC_GRUPO_CARTAO_WPP AS STRING) AS DSC_GRUPO_CARTAO_WPP,
        CAST(FLAG_SOS AS INT) AS FLAG_SOS,
        CAST(VALOR_SOS AS INT) AS VALOR_SOS
    FROM df_base_recarga
""")


#  Tabela recarga (inserir dados dos csv antes de unir com a tabela principal)

In [ ]:
# carregar tabela recarga
df_base_recarga = spark.read.parquet(PATH_BASE_RECARGA)
df_base_recarga.createOrReplaceTempView("df_base_recarga")

In [ ]:
# Mudando o datatype
df_base_recarga = spark.sql("""
    SELECT
        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        CAST(DW_NUM_NTC AS STRING) AS DW_NUM_NTC,
        CAST(DW_NUM_CLIENTE AS STRING) AS DW_NUM_CLIENTE,
        CAST(DAT_INSERCAO_CREDITO AS STRING) AS DAT_INSERCAO_CREDITO,
        CAST(HOR_INSERCAO_CREDITO AS STRING) AS HOR_INSERCAO_CREDITO,
        CAST(COD_TECNOLOGIA_DW AS STRING) AS COD_TECNOLOGIA_DW,
        CAST(COD_CANAL_AQUISICAO AS INT) AS COD_CANAL_AQUISICAO,
        CAST(COD_TIPO_CREDITO AS STRING) AS COD_TIPO_CREDITO,
        CAST(COD_PROMOCAO AS INT) AS COD_PROMOCAO,
        CAST(VAL_CREDITO_INSERIDO AS FLOAT) AS VAL_CREDITO_INSERIDO,
        CAST(VAL_BONUS AS FLOAT) AS VAL_BONUS,
        CAST(VAL_REAL AS FLOAT) AS VAL_REAL,
        CAST(COD_PLATAFORMA_ATU AS STRING) AS COD_PLATAFORMA_ATU,
        CAST(COD_STATUS_PLATAFORMA AS STRING) AS COD_STATUS_PLATAFORMA,
        CAST(IND_METODO_PAGAMENTO AS STRING) AS IND_METODO_PAGAMENTO,
        CAST(DW_PLANO_TARIFACAO AS INT) AS DW_PLANO_TARIFACAO,
        CAST(DW_TIPO_RECARGA AS INT) AS DW_TIPO_RECARGA,
        CAST(DW_TIPO_INSERCAO AS INT) AS DW_TIPO_INSERCAO,
        CAST(DW_FORMA_PAGAMENTO AS INT) AS DW_FORMA_PAGAMENTO,
        CAST(DW_INSTITUICAO AS INT) AS DW_INSTITUICAO,
        CAST(COD_GRUPO_CARTAO AS STRING) AS COD_GRUPO_CARTAO,
        CAST(DSC_GRUPO_CARTAO_WPP AS STRING) AS DSC_GRUPO_CARTAO_WPP,
        CAST(FLAG_SOS AS INT) AS FLAG_SOS,
        CAST(VALOR_SOS AS INT) AS VALOR_SOS
    FROM df_base_recarga
""")

In [ ]:
# join da tabela recarga com df_CANAL_AQUISICAO_CREDITO
df_base_recarga.createOrReplaceTempView("df_base_recarga")
df_CANAL_AQUISICAO_CREDITO.createOrReplaceTempView("CANAL_AQUISICAO_CREDITO")
df_base_recarga_canal = spark.sql("""
    SELECT
        f.*,
        c.COD_CANAL_AQUISICAO AS CAN_COD_CANAL_AQUISICAO,
        c.DSC_CANAL_AQUISICAO AS CAN_DSC_CANAL_AQUISICAO,
        c.COD_SISTEMA_DW AS CAN_COD_SISTEMA_DW,
        c.DAT_ATUALIZACAO_DW AS CAN_DAT_ATUALIZACAO_DW,
        c.DAT_CRIACAO_DW AS CAN_DAT_CRIACAO_DW,
        c.COD_CANAL_AQUISICAO_BI AS CAN_COD_CANAL_AQUISICAO_BI,
        c.DSC_CANAL_AQUISICAO_BI AS CAN_DSC_CANAL_AQUISICAO_BI,
        c.COD_AGENTE_CREDITO AS CAN_COD_AGENTE_CREDITO,
        c.DAT_EXPIRACAO_DW AS CAN_DAT_EXPIRACAO_DW,
        c.COD_TIPO_CREDITO AS CAN_COD_TIPO_CREDITO,
        c.COD_TIPO_INSTITUICAO AS CAN_COD_TIPO_INSTITUICAO,
        c.DSC_TIPO_INSTITUICAO AS CAN_DSC_TIPO_INSTITUICAO
    FROM df_base_recarga f
    LEFT JOIN CANAL_AQUISICAO_CREDITO c
        ON f.COD_CANAL_AQUISICAO = c.COD_CANAL_AQUISICAO
""")


In [ ]:
# FORMA_PAGAMENTO
df_base_recarga_canal.createOrReplaceTempView("df_base_recarga_canal")
df_FORMA_PAGAMENTO.createOrReplaceTempView("FORMA_PAGAMENTO")
df_base_recarga_forma = spark.sql("""
    SELECT
        f.*,
        fp.COD_FORMA_PAGAMENTO AS FP_COD_FORMA_PAGAMENTO,
        fp.DSC_FORMA_PAGAMENTO AS FP_DSC_FORMA_PAGAMENTO,
        fp.DAT_CRIACAO_DW AS FP_DAT_CRIACAO_DW,
        fp.DAT_EXPIRACAO_DW AS FP_DAT_EXPIRACAO_DW
    FROM df_base_recarga_canal f
    LEFT JOIN FORMA_PAGAMENTO fp
        ON f.DW_FORMA_PAGAMENTO = fp.DW_FORMA_PAGAMENTO
""")



In [ ]:
#instituição
df_base_recarga_forma.createOrReplaceTempView("df_base_recarga_forma")
df_INSTITUICAO.createOrReplaceTempView("INSTITUICAO")
df_base_recarga_instituicao = spark.sql("""
    SELECT
        f.*,
        i.COD_INSTITUICAO AS INS_COD_INSTITUICAO,
        i.DSC_INSTITUICAO AS INS_DSC_INSTITUICAO,
        i.COD_TIPO_INSTITUICAO AS INS_COD_TIPO_INSTITUICAO,
        i.DSC_TIPO_INSTITUICAO AS INS_DSC_TIPO_INSTITUICAO,
        i.COD_SISTEMA_DW AS INS_COD_SISTEMA_DW,
        i.DAT_EXPIRACAO_DW AS INS_DAT_EXPIRACAO_DW,
        i.DAT_CRIACAO_DW AS INS_DAT_CRIACAO_DW,
        i.COD_AGENTE AS INS_COD_AGENTE
    FROM df_base_recarga_forma f
    LEFT JOIN INSTITUICAO i
        ON f.DW_INSTITUICAO = i.DW_INSTITUICAO
""")

In [ ]:
# Plano Preço
df_base_recarga_instituicao.createOrReplaceTempView("df_base_recarga_instituicao")
df_PLANO_PRECO.createOrReplaceTempView("PLANO_PRECO")
df_base_recarga_plano = spark.sql("""
    SELECT
        f.*,
        pp.DW_PLANO AS PP_DW_PLANO,
        pp.COD_PLANO_PRECO AS PP_COD_PLANO_PRECO,
        pp.DSC_PLANO_PRECO AS PP_DSC_PLANO_PRECO,
        pp.COD_TIPO_CLIENTE AS PP_COD_TIPO_CLIENTE,
        pp.COD_SUB_TIPO_CLIENTE AS PP_COD_SUB_TIPO_CLIENTE,
        pp.DW_TIPO_CLIENTE AS PP_DW_TIPO_CLIENTE,
        pp.DAT_EFETIVACAO AS PP_DAT_EFETIVACAO,
        pp.DAT_EXPIRACAO AS PP_DAT_EXPIRACAO,
        pp.DSC_PLANO_PRECO_BI AS PP_DSC_PLANO_PRECO_BI,
        pp.DSC_GRUPO_PLANO_BI AS PP_DSC_GRUPO_PLANO_BI,
        pp.DSC_TIPO_PLANO_BI AS PP_DSC_TIPO_PLANO_BI,
        pp.IND_AMDOCS_PLAT_PRE AS PP_IND_AMDOCS_PLAT_PRE,
        pp.COD_TRATAMENTO_ESPECIAL AS PP_COD_TRATAMENTO_ESPECIAL,
        pp.COD_SISTEMA_DW AS PP_COD_SISTEMA_DW,
        pp.COD_TECNOLOGIA_DW AS PP_COD_TECNOLOGIA_DW,
        pp.DAT_EXPIRACAO_DW AS PP_DAT_EXPIRACAO_DW,
        pp.DAT_ATUALIZACAO_DW AS PP_DAT_ATUALIZACAO_DW,
        pp.DAT_CRIACAO_DW AS PP_DAT_CRIACAO_DW,
        pp.NUM_FRANQUIA_MINUTOS_BI AS PP_NUM_FRANQUIA_MINUTOS_BI,
        pp.NUM_FRANQUIA_REAIS_BI AS PP_NUM_FRANQUIA_REAIS_BI,
        pp.NUM_FRANQUIA_EVENTOS_BI AS PP_NUM_FRANQUIA_EVENTOS_BI,
        pp.NUM_FRANQUIA_VOLUME_BI AS PP_NUM_FRANQUIA_VOLUME_BI,
        pp.COD_PLANO_COMPONENTE AS PP_COD_PLANO_COMPONENTE,
        pp.DSC_PLANO_PRECO_UNICO_BI AS PP_DSC_PLANO_PRECO_UNICO_BI,
        pp.DSC_MODALIDADE_PLANO AS PP_DSC_MODALIDADE_PLANO
    FROM df_base_recarga_instituicao f
    LEFT JOIN PLANO_PRECO pp
        ON CAST(f.DW_PLANO_TARIFACAO AS STRING) = CAST(pp.COD_PLANO_PRECO AS STRING)
""")

In [ ]:
# Plataforma
df_base_recarga_plano.createOrReplaceTempView("df_base_recarga_plano")
df_STATUS_PLATAFORMA.createOrReplaceTempView("STATUS_PLATAFORMA")
df_base_plataforma = spark.sql("""
    SELECT
        f.*,
        sp.DSC_STATUS_PLATAFORMA AS SP_DSC_STATUS_PLATAFORMA,
        sp.IND_ATIVO AS SP_IND_ATIVO,
        sp.DAT_ATUALIZACAO_DW AS SP_DAT_ATUALIZACAO_DW,
        sp.DAT_CRIACAO_DW AS SP_DAT_CRIACAO_DW,
        sp.COD_STATUS_PLAT_GRP AS SP_COD_STATUS_PLAT_GRP,
        sp.IND_STS_PLAT_GRP_ATIVO AS SP_IND_STS_PLAT_GRP_ATIVO
    FROM df_base_recarga_plano f
    LEFT JOIN STATUS_PLATAFORMA sp
        ON f.COD_STATUS_PLATAFORMA = sp.COD_STATUS_PLATAFORMA
""")

In [ ]:
# Promoção
df_base_plataforma.createOrReplaceTempView("df_plataforma")
df_PROMOCAO_CREDITO.createOrReplaceTempView("PROMOCAO_CREDITO")
df_base_recarga_promocao = spark.sql("""
    SELECT
        f.*,
        pr.DSC_PROMOCAO AS PR_DSC_PROMOCAO,
        pr.DAT_EXPIRACAO_DW AS PR_DAT_EXPIRACAO_DW,
        pr.DAT_ATUALIZACAO_DW AS PR_DAT_ATUALIZACAO_DW,
        pr.DAT_CRIACAO_DW AS PR_DAT_CRIACAO_DW,
        pr.COD_PROM_GRUPO_CARTAO AS PR_COD_PROM_GRUPO_CARTAO,
        pr.DSC_NOME_PROMOCAO AS PR_DSC_NOME_PROMOCAO,
        pr.COD_TIPO_PROMOCAO AS PR_COD_TIPO_PROMOCAO,
        pr.DAT_INICIO_VIGENCIA AS PR_DAT_INICIO_VIGENCIA,
        pr.DAT_FIM_VIGENCIA AS PR_DAT_FIM_VIGENCIA,
        pr.VAL_PROMOCAO AS PR_VAL_PROMOCAO,
        pr.NUM_CONTA_DEDICADA AS PR_NUM_CONTA_DEDICADA
    FROM df_plataforma f
    LEFT JOIN PROMOCAO_CREDITO pr
        ON f.COD_PROMOCAO = pr.COD_PROMOCAO
""")

In [ ]:
# tecnologia
df_base_recarga_promocao.createOrReplaceTempView("df_base_recarga_promocao")
df_TECNOLOGIA.createOrReplaceTempView("TECNOLOGIA")
df_base_recarga_tecnologia = spark.sql("""
    SELECT
        f.*,
        t.DSC_TECNOLOGIA AS TEC_DSC_TECNOLOGIA,
        t.DAT_ATUALIZACAO_DW AS TEC_DAT_ATUALIZACAO_DW,
        t.DAT_CRIACAO_DW AS TEC_DAT_CRIACAO_DW,
        t.COD_TECNOLOGIA_SVA AS TEC_COD_TECNOLOGIA_SVA
    FROM df_base_recarga_promocao f
    LEFT JOIN TECNOLOGIA t
        ON CAST(f.COD_TECNOLOGIA_DW AS STRING) = CAST(t.COD_TECNOLOGIA_DW AS STRING)
""")

In [ ]:
#tipo credito
df_base_recarga_tecnologia.createOrReplaceTempView("df_base_recarga_tecnologia_temp") # Use a temp view name to avoid conflicts
df_TIPO_CREDITO.createOrReplaceTempView("TIPO_CREDITO")

# Perform the join using the renamed fact column
df_base_recarga_tipocredito = spark.sql("""
    SELECT
        f.*,
        tc.COD_TIPO_CREDITO AS TPC_COD_TIPO_CREDITO,
        tc.DSC_TIPO_CREDITO AS TPC_DSC_TIPO_CREDITO,
        tc.DAT_EXPIRACAO_DW AS TPC_DAT_EXPIRACAO_DW,
        tc.DAT_ATUALIZACAO_DW AS TPC_DAT_ATUALIZACAO_DW,
        tc.DAT_CRIACAO_DW AS TPC_DAT_CRIACAO_DW
    FROM df_base_recarga_tecnologia_temp f
    LEFT JOIN TIPO_CREDITO tc
        ON f.COD_TIPO_CREDITO = tc.COD_TIPO_CREDITO
""")

In [ ]:
# tipo inserção
df_base_recarga_tipocredito.createOrReplaceTempView("df_base_recarga_tipocredito")
df_TIPO_INSERCAO.createOrReplaceTempView("TIPO_INSERCAO")

df_base_recarga_tipoinsercao = spark.sql("""

       SELECT
        f.*,

        -- colunas da dimensão TIPO_INSERCAO com prefixo TPI_
        ti.DW_TIPO_INSERCAO  AS TPI_DW_TIPO_INSERCAO,
        ti.DSC_TIPO_INSERCAO AS TPI_DSC_TIPO_INSERCAO,
        ti.DAT_EXPIRACAO_DW  AS TPI_DAT_EXPIRACAO_DW,
        ti.DAT_CRIACAO_DW    AS TPI_DAT_CRIACAO_DW

    FROM df_base_recarga_tipocredito f
    LEFT JOIN TIPO_INSERCAO ti
        ON f.DW_TIPO_INSERCAO = ti.DW_TIPO_INSERCAO
""")

In [ ]:
#tipo recarga
df_base_recarga_tipoinsercao.createOrReplaceTempView("df_base_recarga_tipoinsercao")
df_TIPO_RECARGA.createOrReplaceTempView("TIPO_RECARGA")

df_bases_recarga_tiporecarga = spark.sql("""

       SELECT
        f.*,

        -- colunas da dimensão TIPO_RECARGA com prefixo TR_
        tr.DW_TIPO_RECARGA  AS TR_DW_TIPO_RECARGA,
        tr.DSC_TIPO_RECARGA AS TR_DSC_TIPO_RECARGA,
        tr.DAT_EXPIRACAO_DW AS TR_DAT_EXPIRACAO_DW,
        tr.DAT_CRIACAO_DW   AS TR_DAT_CRIACAO_DW

    FROM df_base_recarga_tipoinsercao f
    LEFT JOIN TIPO_RECARGA tr
        ON f.DW_TIPO_RECARGA = tr.DW_TIPO_RECARGA
""")





In [ ]:
# criando coluna SAFRA
df_bases_recarga_tiporecarga.createOrReplaceTempView("df_bases_recarga_tiporecarga")

df_bases_recarga = spark.sql("""
SELECT
    f.*,
    trunc(
        to_timestamp(f.DAT_INSERCAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),
        'MM'
    ) AS SAFRA
FROM df_bases_recarga_tiporecarga f
""")

# colocar indentificador nas variaveis da tabela book_pagamento
for c in df_bases_recarga.columns:
    df_bases_recarga = df_bases_recarga.withColumnRenamed(
        c, f"B_RECARGA_{c}"
    )


In [ ]:
df_bases_recarga.show(5)

+-----------------+--------------------+------------------------+------------------------------+------------------------------+---------------------------+-----------------------------+--------------------------+----------------------+------------------------------+-------------------+------------------+----------------------------+-------------------------------+------------------------------+----------------------------+-------------------------+--------------------------+----------------------------+------------------------+--------------------------+------------------------------+------------------+-------------------+---------------------------------+---------------------------------+----------------------------+--------------------------------+----------------------------+------------------------------------+------------------------------------+--------------------------------+------------------------------+------------------------------+----------------------------------+-------

In [ ]:
# Unir tabelas
df_principal_telco.createOrReplaceTempView("tabela_1")
df_bases_recarga.createOrReplaceTempView("df_bases_recarga")

df_principal_recarga = spark.sql("""
SELECT
    -- todas as colunas da tabela 1
    b1.*,

    -- todas as colunas da tabela 2
    r2.*

FROM tabela_1 b1
LEFT JOIN df_bases_recarga r2
    ON b1.NUM_CPF        = r2.B_RECARGA_NUM_CPF
   AND b1.B_BUREAU_SAFRA = r2.B_RECARGA_SAFRA
""")

# Unir tabela Principal com a tabela book_pagamento

Book_pagamento possui arquivo em excel contendo informações dos metadados

Analise posterior feita pela equipe de Ciência e Analise de dados

In [ ]:
# carregando a tabela book pagamento
df_book_pagamento = spark.read.parquet(PATH_BASE_BOOK_PAGAMENTO)
df_book_pagamento.createOrReplaceTempView("df_book_pagamento")

In [ ]:
# Alterando datatype da tabela
df_book_pagamento.createOrReplaceTempView("df_book_pagamento")

df_book_pagamento = spark.sql("""

SELECT
CAST(NUM_CPF AS STRING) AS NUM_CPF,
TO_DATE(DAT_STATUS_FATURA, 'ddMMMyyyy:HH:mm:ss') AS DAT_STATUS_FATURA,
CAST(CONTRATO AS STRING) AS CONTRATO,
CAST(SEQ_FATURA AS INT) AS SEQ_FATURA,
CAST(NUM_SUB_SEQ_FATURA AS INT) AS NUM_SUB_SEQ_FATURA,
CAST(NUM_CREDITO_SEQ AS INT) AS NUM_CREDITO_SEQ,
CAST(DW_TIPO_FATURA AS INT) AS DW_TIPO_FATURA,
CAST(IND_STATUS_FATURA AS STRING) AS IND_STATUS_FATURA,
CAST(DW_NUM_CLIENTE AS STRING) AS DW_NUM_CLIENTE,
CAST(DW_AREA AS INT) AS DW_AREA,
CAST(DW_UN_NEGOCIO AS INT) AS DW_UN_NEGOCIO,
CAST(DW_FORMA_PAGAMENTO AS INT) AS DW_FORMA_PAGAMENTO,
CAST(VAL_PAGAMENTO_FATURA AS FLOAT) AS VAL_PAGAMENTO_FATURA,
TO_DATE(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss') AS DAT_CRIACAO_DW,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_CRIACAO_DW,
CAST(DW_BANCO AS STRING) AS DW_BANCO,
CAST(DW_TIPO_PAGAMENTO AS STRING) AS DW_TIPO_PAGAMENTO,
CAST(NUM_BANCO_PAGAMENTO AS STRING) AS NUM_BANCO_PAGAMENTO,
CAST(NUM_AGENCIA_PAGAMENTO AS STRING) AS NUM_AGENCIA_PAGAMENTO,
CAST(NUM_CC_PAGAMENTO AS STRING) AS NUM_CC_PAGAMENTO,
CAST(DW_MOTIVO_ESTORNO AS STRING) AS DW_MOTIVO_ESTORNO,
CAST(VAL_DESCONTO_ITEM AS FLOAT) AS VAL_DESCONTO_ITEM,
CAST(VAL_PAGAMENTO_ITEM AS FLOAT) AS VAL_PAGAMENTO_ITEM,
CAST(VAL_JUROS_MULTAS_ITEM AS FLOAT) AS VAL_JUROS_MULTAS_ITEM,
CAST(VAL_MULTA_EQUIP_ITEM AS FLOAT) AS VAL_MULTA_EQUIP_ITEM,
CAST(VAL_MULTA_EQUIP_TOTAL AS FLOAT) AS VAL_MULTA_EQUIP_TOTAL,
CAST(VAL_MULTA_FID_ITEM AS FLOAT) AS VAL_MULTA_FID_ITEM,
CAST(COD_ORIGEM_NETUNO AS STRING) AS COD_ORIGEM_NETUNO,
CAST(COD_CONTA_ATIVIDADE AS STRING) AS COD_CONTA_ATIVIDADE,
CAST(SEQ_ENTIDADE_ATIVIDADE AS INT) AS SEQ_ENTIDADE_ATIVIDADE,
TO_DATE(DAT_CRIACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss')                              AS DAT_CRIACAO_ATIVIDADE,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')     AS HR_CRIACAO_ATIVIDADE,
TO_DATE(DAT_ATUALIZACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss')                          AS DAT_ATUALIZACAO_ATIVIDADE,
TO_CHAR(TO_TIMESTAMP(DAT_ATUALIZACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_ATUALIZACAO_ATIVIDADE,
CAST(COD_LOGIN_OPERADOR_ATIVIDADE AS STRING) AS COD_LOGIN_OPERADOR_ATIVIDADE,
CAST(COD_ATIVIDADE AS STRING) AS COD_ATIVIDADE,
CAST(COD_RAZAO_ATIVIDADE AS STRING) AS COD_RAZAO_ATIVIDADE,
TO_DATE(DAT_BAIXA_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss') AS DAT_BAIXA_ATIVIDADE,
CAST(VAL_BAIXA_ATIVIDADE AS FLOAT) AS VAL_BAIXA_ATIVIDADE,
TO_DATE(DAT_DEPOSITO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss') AS DAT_DEPOSITO_ATIVIDADE,
CAST(COD_FUNDO_ATIVIDADE AS STRING) AS COD_FUNDO_ATIVIDADE,
CAST(COD_BANCO_ATIVIDADE AS STRING) AS COD_BANCO_ATIVIDADE,
CAST(NUM_CONTA_ATIVIDADE AS STRING) AS NUM_CONTA_ATIVIDADE,
CAST(COD_AGENCIA_ATIVIDADE AS STRING) AS COD_AGENCIA_ATIVIDADE,
CAST(SEQ_ENTIDADE_PAGAMENTO AS INT) AS SEQ_ENTIDADE_PAGAMENTO,
TO_DATE(DAT_CRIACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss')                              AS DAT_CRIACAO_PAGAMENTO,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss')     AS HR_CRIACAO_PAGAMENTO,
TO_DATE(DAT_ATUALIZACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss')                          AS DAT_ATUALIZACAO_PAGAMENTO,
TO_CHAR(TO_TIMESTAMP(DAT_ATUALIZACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_ATUALIZACAO_PAGAMENTO,
CAST(COD_LOGIN_PAGAMENTO AS STRING) AS COD_LOGIN_PAGAMENTO,
CAST(COD_FORMA_PAGAMENTO AS STRING) AS COD_FORMA_PAGAMENTO,
CAST(VAL_ORIGINAL_PAGAMENTO AS FLOAT) AS VAL_ORIGINAL_PAGAMENTO,
CAST(NUM_FATURA_PAGAMENTO AS STRING) AS NUM_FATURA_PAGAMENTO,
CAST(COD_TIPO_PAGAMENTO AS STRING) AS COD_TIPO_PAGAMENTO,
CAST(DSC_NOME_BANCO_PAGAMENTO AS STRING) AS DSC_NOME_BANCO_PAGAMENTO,
CAST(SEQ_ARQUIVO_PAGAMENTO AS INT) AS SEQ_ARQUIVO_PAGAMENTO,
CAST(NUM_PARCELA_PAGAMENTO AS INT) AS NUM_PARCELA_PAGAMENTO,
CAST(NUM_AGRUPADOR_PAGAMENTO AS INT) AS NUM_AGRUPADOR_PAGAMENTO,
CAST(DSC_PAGAMENTO AS STRING) AS DSC_PAGAMENTO,
CAST(VAL_ATUAL_PAGAMENTO AS FLOAT) AS VAL_ATUAL_PAGAMENTO,
CAST(COD_METODO_PAGAMENTO AS INT) AS COD_METODO_PAGAMENTO,
CAST(IND_STATUS_PAGAMENTO AS STRING) AS IND_STATUS_PAGAMENTO,
TO_DATE(DAT_STATUS_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss') AS DAT_STATUS_PAGAMENTO,
CAST(COD_ARQUIVO_PAGAMENTO AS STRING) AS COD_ARQUIVO_PAGAMENTO,
CAST(COD_NETUNO_PAGAMENTO AS STRING) AS COD_NETUNO_PAGAMENTO,
TO_DATE(DAT_CRIACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss')                          AS DAT_CRIACAO_CREDITO,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),'HH:mm:ss') AS HR_CRIACAO_CREDITO,
TO_DATE(DAT_ATUALIZACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss')                      AS DAT_ATUALIZACAO_CREDITO,
CAST(COD_LOGIN_CREDITO AS STRING) AS COD_LOGIN_CREDITO,
CAST(VAL_PAGAMENTO_CREDITO AS FLOAT) AS VAL_PAGAMENTO_CREDITO,
CAST(IND_TIPO_CREDITO AS STRING) AS IND_TIPO_CREDITO,
CAST(SEQ_PAGAMENTO_CREDITO AS INT) AS SEQ_PAGAMENTO_CREDITO,
CAST(SEQ_FATURA_CREDITO AS INT) AS SEQ_FATURA_CREDITO,
CAST(COD_ALOCACAO_CREDITO AS STRING) AS COD_ALOCACAO_CREDITO,
CAST(COD_DESALOCACAO_CREDITO AS STRING) AS COD_DESALOCACAO_CREDITO,
CAST(SEQ_ENTIDADE_CREDITO AS INT) AS SEQ_ENTIDADE_CREDITO,
CAST(COD_TIPO_FATURA AS STRING) AS COD_TIPO_FATURA,
TO_DATE(DAT_ATIVIDADE_CREDITO, 'ddMMMyyyy:HH:mm:ss') AS DAT_ATIVIDADE_CREDITO,
TO_DATE(DAT_VENCIMENTO_CREDITO, 'ddMMMyyyy:HH:mm:ss') AS DAT_VENCIMENTO_CREDITO

FROM df_book_pagamento
""")

NameError: name 'df_book_pagamento' is not defined

In [ ]:
#Criando coluna safra
df_book_pagamento.createOrReplaceTempView("df_book_pagamento")

df_book_pagamento_01 = spark.sql("""
SELECT
    *,
    TO_DATE(
        CONCAT(
            YEAR(DAT_STATUS_FATURA), '-',
            LPAD(MONTH(DAT_STATUS_FATURA), 2, '0'), '-01'
        )
    ) AS SAFRA
FROM df_book_pagamento
""")

# colocar indentificador nas variaveis da tabela book_pagamento
for c in df_book_pagamento_01.columns:
    df_book_pagamento_01 = df_book_pagamento_01.withColumnRenamed(
        c, f"BOOK_PGTO_{c}"
    )
df_book_pagamento = df_book_pagamento_01

In [ ]:
# tabelas temporárias
df_principal_recarga.createOrReplaceTempView("tabela_1")
df_book_pagamento.createOrReplaceTempView("tabela_2")

df_principal_book = spark.sql("""
SELECT
    -- todas as colunas da tabela 1
    b1.*,

    -- todas as colunas da tabela 2
    r2.*

FROM tabela_1 b1
LEFT JOIN tabela_2 r2
    ON b1.NUM_CPF        = r2.BOOK_PGTO_NUM_CPF
   AND b1.B_BUREAU_SAFRA = r2.BOOK_PGTO_SAFRA
""")

In [ ]:
df_principal_book.printSchema()

# Unir tabela principal com tabela book_atraso

In [ ]:
# carregando a tabela book atraso
df_book_atraso = spark.read.parquet(PATH_BASE_BOOK_ATRASO)
df_book_atraso.createOrReplaceTempView("df_book_atraso")

In [ ]:
# Alterando o datatype
df_book_atraso = spark.sql("""
SELECT
    CAST(NUM_CPF AS STRING) AS NUM_CPF,
    DAT_REFERENCIA AS DAT_REFERENCIA,
    CAST(NUM_FATURA_HASH AS STRING) AS NUM_FATURA_HASH,
    CAST(NUM_ENT_SEQ_FATURA AS INT) AS NUM_ENT_SEQ_FATURA,
    CAST(CONTRATO AS BIGINT) AS CONTRATO,
    CAST(DW_UN_NEGOCIO AS INT) AS DW_UN_NEGOCIO,
    CAST(DW_HIS_PONTO_VENDA_COMTA AS BIGINT) AS DW_HIS_PONTO_VENDA_COMTA,
    CAST(DW_NUM_CLIENTE AS BIGINT) AS DW_NUM_CLIENTE,
    CAST(DW_AREA AS INT) AS DW_AREA,
    CAST(DW_CICLO AS INT) AS DW_CICLO,
    CAST(DW_TIPO_CLIENTE_CONTA AS INT) AS DW_TIPO_CLIENTE_CONTA,
    CAST(DW_OFERTA AS INT) AS DW_OFERTA,
    CAST(DW_FAIXA_AGING_FATURA AS INT) AS DW_FAIXA_AGING_FATURA,
    CAST(DW_FAIXA_AGING_DIVIDA AS INT) AS DW_FAIXA_AGING_DIVIDA,
    CAST(DW_FAIXA_TEMPO_BASE AS INT) AS DW_FAIXA_TEMPO_BASE,
    CAST(DW_FAIXA_AGING_PROX_FECH AS INT) AS DW_FAIXA_AGING_PROX_FECH,
    CAST(DW_TIPO_FATURAMENTO AS INT) AS DW_TIPO_FATURAMENTO,
    CAST(COD_PLATAFORMA AS STRING) AS COD_PLATAFORMA,

    -- todas as DAT_* como STRING
    CAST(DAT_CRIACAO_REGISTRO_TRANS AS STRING) AS DAT_CRIACAO_REGISTRO_TRANS,
    CAST(DAT_ALTERACAO_REGISTRO_TRANS AS STRING) AS DAT_ALTERACAO_REGISTRO_TRANS,
    CAST(DAT_CANCELAMENTO_FAT AS STRING) AS DAT_CANCELAMENTO_FAT,
    CAST(DAT_ORIGINAL_VCTO_FAT AS STRING) AS DAT_ORIGINAL_VCTO_FAT,
    CAST(DAT_ALTERACAO_VCTO_FAT AS STRING) AS DAT_ALTERACAO_VCTO_FAT,
    CAST(DAT_CRIACAO_FAT AS STRING) AS DAT_CRIACAO_FAT,
    CAST(DAT_VENCIMENTO_FAT AS STRING) AS DAT_VENCIMENTO_FAT,
    CAST(DAT_STATUS_FAT AS STRING) AS DAT_STATUS_FAT,
    CAST(DAT_MIN_VENCIMENTO_FAT AS STRING) AS DAT_MIN_VENCIMENTO_FAT,
    CAST(DAT_ATIVACAO_CONTA_CLI AS STRING) AS DAT_ATIVACAO_CONTA_CLI,
    CAST(DAT_CRIACAO_DW AS STRING) AS DAT_CRIACAO_DW,

    CAST(NUM_BILL_SEQ_FAT AS INT) AS NUM_BILL_SEQ_FAT,
    CAST(NUM_SEQ_ACORDO_FAT AS INT) AS NUM_SEQ_ACORDO_FAT,

    CAST(IND_ISENCAO_COB_FAT AS STRING) AS IND_ISENCAO_COB_FAT,
    CAST(IND_WO AS STRING) AS IND_WO,
    CAST(IND_PDD AS STRING) AS IND_PDD,
    CAST(IND_PCCR AS STRING) AS IND_PCCR,
    CAST(IND_ACA AS STRING) AS IND_ACA,
    CAST(IND_PRIMEIRA_FAT AS STRING) AS IND_PRIMEIRA_FAT,
    CAST(IND_FRAUDE AS STRING) AS IND_FRAUDE,

    CAST(VAL_FAT_LIQUIDO AS DECIMAL(18,2)) AS VAL_FAT_LIQUIDO,
    CAST(VAL_FAT_BRUTO AS DECIMAL(18,2)) AS VAL_FAT_BRUTO,
    CAST(VAL_FAT_CREDITO AS DECIMAL(18,2)) AS VAL_FAT_CREDITO,
    CAST(VAL_FAT_AJUSTE AS DECIMAL(18,2)) AS VAL_FAT_AJUSTE,
    CAST(VAL_FAT_BRUTO_BC AS DECIMAL(18,2)) AS VAL_FAT_BRUTO_BC,
    CAST(VAL_FAT_PAGAMENTO_BRUTO AS DECIMAL(18,2)) AS VAL_FAT_PAGAMENTO_BRUTO,
    CAST(VAL_FAT_ABERTO AS DECIMAL(18,2)) AS VAL_FAT_ABERTO,
    CAST(VAL_FAT_ABERTO_LIQ AS DECIMAL(18,2)) AS VAL_FAT_ABERTO_LIQ,
    CAST(VAL_MULTA_JUROS AS DECIMAL(18,2)) AS VAL_MULTA_JUROS,
    CAST(VAL_MULTA_CANCELAMENTO AS DECIMAL(18,2)) AS VAL_MULTA_CANCELAMENTO,
    CAST(VAL_PARC_APARELHO_LIQ AS DECIMAL(18,2)) AS VAL_PARC_APARELHO_LIQ,
    CAST(VAL_FAT_LIQ_JM_MC AS DECIMAL(18,2)) AS VAL_FAT_LIQ_JM_MC
FROM df_book_atraso
""")


In [ ]:
#carregando cvs do book atraso
df_tipo_faturamento = spark.read.csv(PATH_DIMENSOES_BOOK_ATRASO, header=True, sep=",", inferSchema=True)
df_tipo_faturamento.createOrReplaceTempView("df_tipo_faturamento")

In [ ]:
# Unir os dados do csv BI_DIM_TIPO_FATURAMENTO com os dados do book atraso
# Criar views temporárias
df_tipo_faturamento.createOrReplaceTempView("df_tipo_faturamento")
df_book_atraso.createOrReplaceTempView("BOOK_ATRASO")

# Query ajustada para renomear a coluna DAT_CRIACAO_DW do df_tipo_faturamento
df_book_atraso = spark.sql(
    """
    SELECT
        f.*,
        b.DSC_TIPO_FATURAMENTO,
        b.COD_TIPO_FATURAMENTO,
        b.DAT_EXPIRACAO_DW AS DIM_DAT_EXPIRACAO_DW, -- Renaming to avoid conflict
        b.DAT_CRIACAO_DW AS DIM_DAT_CRIACAO_DW,   -- Renaming to avoid conflict
        b.DSC_TIPO_FATURAMENTO_ABREV
    FROM BOOK_ATRASO f
    LEFT JOIN df_tipo_faturamento b
        ON f.DW_TIPO_FATURAMENTO = b.DW_TIPO_FATURAMENTO
    """
)



In [ ]:
#Criando coluna SAFRA, VAI SER USADO OS DADOS DA COLUNA "DAT_REFERENCIA",
#PODE SER MUDADO DE ACORDO COM A NECESSIDADE DO NEGOCIO
df_book_atraso.createOrReplaceTempView("df_book_atraso")

df_book_atraso = spark.sql("""
SELECT
    *,
    TO_DATE(
        TO_TIMESTAMP(
            UPPER(DAT_REFERENCIA),
            'ddMMMyyyy:HH:mm:ss'
        )
    ) AS SAFRA,
    YEAR(
        TO_TIMESTAMP(
            UPPER(DAT_REFERENCIA),
            'ddMMMyyyy:HH:mm:ss'
        )
    ) AS SAFRA_ANO,
    MONTH(
        TO_TIMESTAMP(
            UPPER(DAT_REFERENCIA),
            'ddMMMyyyy:HH:mm:ss'
        )
    ) AS SAFRA_MES
FROM df_book_atraso
""")

# colocar indentificador nas variaveis da tabela BOOK ATRASO
for c in df_book_atraso.columns:
    df_book_atraso = df_book_atraso.withColumnRenamed(
        c, f"BOOK_ATRASO_{c}"
    )

In [ ]:
# Unir com a tabela df_principal_book e fazer a tabela com todos so dados
# tabelas temporárias
df_principal_book.createOrReplaceTempView("tabela_1")
df_book_atraso.createOrReplaceTempView("tabela_2")

df_principal = spark.sql("""
SELECT
    -- todas as colunas da tabela 1
    b1.*,

    -- todas as colunas da tabela 2
    a2.*

FROM tabela_1 b1
LEFT JOIN tabela_2 a2
    ON b1.NUM_CPF        = a2.BOOK_ATRASO_NUM_CPF
   AND b1.B_BUREAU_SAFRA = a2.BOOK_ATRASO_SAFRA
""")

In [ ]:
# Remover os cpfs das colunas que foram usados como referencia da tabela bureau para fazer o join
df_principal = (
    df_principal
    .drop("BOOK_ATRASO_NUM_CPF")
    .drop("BOOK_PGTO_NUM_CPF")
    .drop("B_RECARGA_NUM_CPF")

)

In [ ]:
df_principal.printSchema()